# RawFileReader Python Adapter — Demo

This notebook shows how to read Thermo Fisher `.raw` mass-spectrometry files
in Google Colab using the **rawfilereader** Python package.

**Steps**
1. Install .NET 8 and pythonnet (cells 1–2)
2. Clone the repo and download the RawFileReader DLLs (cells 3–4)
3. Upload your `.raw` file (cell 5)
4. Explore the file with the adapter API (cells 6+)

> ⚠️ **Run cells in order.** Cell 2 installs pythonnet; if `import clr`
> later raises an error, go to **Runtime → Restart session** and re-run
> from cell 3 onwards (skip cells 1–2).

## Step 1 — Install .NET 8 Runtime

In [ ]:
%%bash
set -e

# Register the Microsoft package feed
wget -q https://packages.microsoft.com/config/ubuntu/$(lsb_release -rs)/packages-microsoft-prod.deb \
     -O packages-microsoft-prod.deb
dpkg -i packages-microsoft-prod.deb > /dev/null
rm packages-microsoft-prod.deb

# Install .NET 8 runtime
apt-get update -qq
apt-get install -y dotnet-runtime-8.0 2>&1 | grep -E '(Setting up|already|error)' || true

echo
echo "--- Installed runtimes ---"
dotnet --list-runtimes

## Step 2 — Install pythonnet

pythonnet must be installed **after** .NET so it links against the correct runtime.

In [ ]:
!pip install -q --force-reinstall pythonnet

# pythonnet defaults to Mono on Linux — force CoreCLR (.NET 8)
import pythonnet
pythonnet.load("coreclr")
import clr
print("pythonnet OK — CLR version:", clr.__version__)

## Step 3 — Clone the repository and install the package

In [ ]:
import os, sys

if not os.path.isdir("/content/RawFileReaderPyAdapter"):
    !git clone -q https://github.com/mzzzhunter/RawFileReaderPyAdapter.git /content/RawFileReaderPyAdapter
else:
    !git -C /content/RawFileReaderPyAdapter pull -q

# Add to sys.path so the current kernel finds the package immediately
# (pip install -e does not update sys.path in an already-running Colab kernel)
repo_path = "/content/RawFileReaderPyAdapter"
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

import rawfilereader
print("rawfilereader", rawfilereader.__version__, "ready.")

In [ ]:
# Automatically reload rawfilereader source files before each cell runs.
# This means a git pull in cell 3 is picked up immediately — no kernel restart needed.
%load_ext autoreload
%autoreload 2

## Step 4 — Download the RawFileReader DLLs

In [ ]:
!python /content/RawFileReaderPyAdapter/download_dlls.py \
    --libs-dir /content/RawFileReaderPyAdapter/libs \
    --no-env

import os
os.environ["RAWFILEREADER_LIBS"] = "/content/RawFileReaderPyAdapter/libs"
print("RAWFILEREADER_LIBS:", os.environ["RAWFILEREADER_LIBS"])

## Step 5 — Upload your `.raw` file

Click the button below to upload a Thermo `.raw` file from your computer.
The path is saved to `RAW_FILE` for use in subsequent cells.

In [ ]:
from google.colab import files

uploaded = files.upload()   # opens the file-picker dialog

RAW_FILE = next(iter(uploaded))   # path of the uploaded file
print(f"Using file: {RAW_FILE}")

## Step 6 — File summary

In [ ]:
from rawfilereader import RawFileAdapter

with RawFileAdapter(RAW_FILE) as rf:
    fi = rf.get_file_info()
    first, last = rf.get_scan_range()

    print(f"File       : {fi.file_name}")
    print(f"Date       : {fi.creation_date}")
    print(f"Operator   : {fi.operator}")
    print(f"Sample     : {fi.sample_name}")
    print(f"Instrument : {fi.instrument_name}  ({fi.instrument_serial_number})")
    print(f"Scans      : {first} – {last}")
    print(f"RT range   : {rf.get_start_time():.2f} – {rf.get_end_time():.2f} min")

## Step 7 — Scan filters present in the file

In [ ]:
with RawFileAdapter(RAW_FILE) as rf:
    filters = rf.get_filters()

print(f"{len(filters)} unique filter(s):")
for f in filters:
    print(" ", f)

## Step 8 — Total Ion Chromatogram (TIC)

In [ ]:
import matplotlib.pyplot as plt

with RawFileAdapter(RAW_FILE) as rf:
    tic = rf.get_chromatogram(trace_type="TIC")

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(tic.times, tic.intensities, linewidth=0.8)
ax.set_xlabel("Retention time (min)")
ax.set_ylabel("Intensity")
ax.set_title("Total Ion Chromatogram")
plt.tight_layout()
plt.show()

## Step 9 — Inspect a single scan

Change `SCAN` to any scan number in the file.

In [ ]:
SCAN = 1   # ← change this

with RawFileAdapter(RAW_FILE) as rf:
    info    = rf.get_scan_info(SCAN)
    centroid = rf.get_centroid_stream(SCAN)

print(f"Scan        : {info.scan_number}")
print(f"MS order    : {info.ms_order}")
print(f"RT          : {info.retention_time:.4f} min")
print(f"Filter      : {info.scan_filter}")
print(f"Inj. time   : {info.injection_time:.2f} ms")
if info.ms_order >= 2:
    print(f"Precursor   : {info.precursor_mass:.4f}  z={info.precursor_charge}")
    print(f"CE          : {info.collision_energy:.1f} eV")
print(f"Peaks       : {len(centroid.masses)}")

# Plot the spectrum
fig, ax = plt.subplots(figsize=(12, 4))
ax.vlines(centroid.masses, 0, centroid.intensities, linewidth=0.6)
ax.set_xlabel("m/z")
ax.set_ylabel("Intensity")
ax.set_title(f"Scan {SCAN}  RT={info.retention_time:.3f} min  [{info.scan_filter}]")
plt.tight_layout()
plt.show()

## Step 10 — Iterate all MS1 scans and collect top peaks

In [ ]:
records = []

with RawFileAdapter(RAW_FILE) as rf:
    for info in rf.iter_scan_info(ms_order=1):
        cd = rf.get_centroid_stream(info.scan_number)
        if cd.peaks:
            top_mass, top_int = max(cd.peaks, key=lambda p: p[1])
        else:
            top_mass, top_int = None, None
        records.append({
            "scan":     info.scan_number,
            "rt":       round(info.retention_time, 4),
            "peaks":    len(cd.masses),
            "top_mz":   round(top_mass, 4) if top_mass else None,
            "top_int":  round(top_int)      if top_int  else None,
        })

print(f"Collected {len(records)} MS1 scans.  First 5:")
for r in records[:5]:
    print(f"  scan={r['scan']}  RT={r['rt']}  peaks={r['peaks']}  "
          f"top_mz={r['top_mz']}  top_int={r['top_int']}")

## Step 11 — Export scan table to CSV

In [ ]:
import csv

out_path = "/content/ms1_scans.csv"
with open(out_path, "w", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=records[0].keys())
    writer.writeheader()
    writer.writerows(records)

print(f"Saved {len(records)} rows to {out_path}")

# Download the CSV to your computer
from google.colab import files
files.download(out_path)